## Agentic Search | Multi-Turn | Tool-use 

The objective is to create an Agent to perform prior-art search over patent corpora.



In [8]:
from huggingface_hub import notebook_login
import os
import json
import chromadb
from chromadb.api.types import Embeddable, EmbeddingFunction
from chromadb.utils import embedding_functions
import asyncio

notebook_login()

In [3]:
# Fields to extract from raw JSON data
RELEVANT_FIELDS = [
    # Identifiers & Linking
    "publication_number",
    "application_number",
    "patent_number",
    # Dates (as epoch ints)
    "date_published",
    "filing_date",
    "patent_issue_date",
    "abandon_date",
    # Status & Classes
    "decision",
    "main_cpc_label",
    "main_ipcr_label",
    # Retrievable Text
    "title",
    "abstract",
    # "claims",  ## The legally enforceable boundaries of the invention — the essence of what’s protected.
    # "summary",
]

def get_IP_data():
    """Load and filter IP data from JSON files, skipping files with decode errors."""
    ip_files = []
    for file in os.listdir("Patent_data"):
        file_path = os.path.join("Patent_data", file)
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                filtered = {key: value for key, value in data.items() if key in RELEVANT_FIELDS}
                ip_files.append(filtered)
        except (UnicodeDecodeError, json.JSONDecodeError) as e:
            print(f"Skipping {file}: {e}")
    return ip_files

patent_data = get_IP_data()

Skipping .DS_Store: 'utf-8' codec can't decode byte 0x80 in position 3131: invalid start byte


#### Designing the multi-turn environment

An episode is “one prior-art search task”.


Initial observation is something like:

You are a patent prior-art search assistant.
Here is a new invention description:
<DESCRIPTION>
You have access to tools:

patent_search(query) – searches in a database of patents (title+abstract).

patent_lookup(patent_id) – shows full details of one patent.
Use the tools as needed, then produce a final ranked list of relevant prior-art patents.


The environment passes this as the initial prompt.

After each tool call, the environment appends the tool output to the conversation and gives that back to the model.

3.2. Actions

    Action = “next model output”. You constrain the format so the model can:

    Call tools:
    e.g. CALL_TOOL patent_search: "solar panel with transparent coating for windows"

    Or produce final answer:
    e.g. FINAL_ANSWER: [US1234..., EP5678..., ...] plus explanation.

The environment will:

    Parse the model’s output.

    If it’s CALL_TOOL, execute the underlying Python function and append result.

    If it’s FINAL_ANSWER, end the episode and compute reward.

This is exactly the pattern used in TRL’s experimental “learning tools” examples with calculator/wiki/python environments. 


3.3. Termination

Terminate when:

    The model outputs FINAL_ANSWER, or

    You hit max_steps (e.g. 5–8 tool calls) and then force the model to give a final answer, or mark as failure.

In [9]:
CHROMA_DB_DIR = ".chroma_db"
_chroma_semaphore: asyncio.Semaphore | None = None

def _get_chroma_semaphore() -> asyncio.Semaphore:
    global _chroma_semaphore
    if _chroma_semaphore is None:
        _chroma_semaphore = asyncio.Semaphore(20)
    return _chroma_semaphore


def create_chroma_collection():
    """Create or get a ChromaDB collection for patent data."""
    chroma_client = chromadb.PersistentClient(path=CHROMA_DB_DIR)
    collection = chroma_client.get_or_create_collection(
        name="patent_collection",
        embedding_function=embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name="sentence-transformers/all-mpnet-base-v2"
        ),
        configuration={
        "hnsw": {
            "space": "cosine",
            "ef_construction": 200,
            "ef_search":150
        }
    }
    )
    collection.add(
        documents=[patent["abstract"] for patent in patent_data],
        ids=[patent["publication_number"] for patent in patent_data],
        metadatas=[{ k:v for k, v in patent.items() if k !='abstract'} 
                   for patent in patent_data],
        )
    
    return collection

collection = create_chroma_collection()


### Defining The Tools

In [5]:
## search tool
async def search_patents(query: str, n_results: int = 10) -> list[dict]:
    """Search for top 10 relevant patents using title embedding similarity."""
    async with _get_chroma_semaphore():
        results = collection.query(
            query_texts=[query],
            n_results=n_results
        )
    
    if not results:
        raise ValueError(f"No results found for query: {query}")
    if not results["metadatas"]:
        raise ValueError(f"No results metadata found for query: {query}")
    
    output = []
    for i in range(len(results["ids"][0])):
        patent_title = results["metadatas"][0][i]["title"]
        publication_number = results["ids"][0][i]
        similarity_score = results["distances"][0][i]
        output.append({"patent_title": patent_title, 
                       "publication_number": publication_number,
                       "similarity_score": similarity_score})
    
    return output

# Patent lookup tool
async def lookup_patent(publication_number: str) -> dict:
    """Lookup patent details by publication number."""
    sem = _get_chroma_semaphore()
    async with sem:
        results = await asyncio.to_thread(
            collection.get,
            ids=[publication_number],
        )

    if not results or not results.get("metadatas"):
        raise ValueError(f"No patent found with publication number: {publication_number}")

    patent_content = results["documents"][0]
    patent_metadata = results["metadatas"][0]
    return {**patent_metadata, "abstract": patent_content}
    

In [7]:
await search_patents("battery management", n_results=3)

[{'patent_title': 'QUICK-EXCHANGE BATTERY ASSEMBLY, AND MOTOR VEHICLE, IN PARTICULAR MOTOR SCOOTER',
  'publication_number': 'US20180151860A1-20180531',
  'similarity_score': 0.5740572214126587},
 {'patent_title': 'METHOD FOR CALCULATING A SETPOINT FOR MANAGING THE FUEL AND ELECTRICITY CONSUMPTION OF A HYBRID MOTOR VEHICLE',
  'publication_number': 'US20180281620A1-20181004',
  'similarity_score': 0.6117575168609619},
 {'patent_title': 'POSITIVE ELECTRODE ACTIVE MATERIAL FOR RECHARGABLE LITHIUM BATTERY, METHOD FOR MANUFACTURING SAME, AND RECHARGABLE LITHIUM BATTERY INCLUDING SAME',
  'publication_number': 'US20180254473A1-20180906',
  'similarity_score': 0.6518891453742981}]

## Reward Design for prior-art Search

### Backward-citation

We need labels for what good prior art looks like.

we are use real patents and their backward citations as ground truth.
For a given patent P, the cited patents = “true” prior art for P.

Then, for each episode:

Target set: ground-truth cited patent IDs for that invention.

Predicted set: IDs the model outputs in FINAL_ANSWER.

### RULER:  LLM-as-a-judge
As in RULER from openpipe


RULER (Relative Universal LLM-Elicited Rewards) is a general-purpose reward function that uses an LLM-as-judge to rank multiple agent trajectories. 

It requires no labeled data, expert feedback, or hand-crafted reward functions, yet reliably improves agent performance.